In [8]:
# Limite les threads BLAS au niveau global aussi (utile même côté parent)
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import sys
from pathlib import Path
SRC = (Path.cwd().parent / "src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np
import seaborn as sns
import warnings
import importlib, params

importlib.reload(params)

warnings.filterwarnings('ignore')

sns.set_theme(style="darkgrid")

In [9]:
PARQUET_PATH = ''

In [12]:
# Load data from SAS pickle file
data = pd.read_parquet(os.path.join(params.DATA_FOLDER, 'wrds\sp500_daily_full.parquet'), 
                       columns=['date', 'permno', 'ret_combined', 'me', 'dollar_vol'])

In [13]:
data.head()

,date,permno,ret_combined,me,dollar_vol
0,2000-01-03,10078,-0.012107,1.194246e+08,1.168195e+09
1,2000-01-03,10104,0.054099,1.681713e+08,2.933259e+09
2,2000-01-03,10107,-0.001606,6.014654e+08,3.139858e+09
3,2000-01-03,10138,-0.049069,4.238815e+06,1.340426e+07
4,2000-01-03,10145,-0.017335,4.473965e+07,1.190890e+08


In [19]:
data_pivot= data[(data['date'] >= params.START_DATE) & (data['date'] <= params.END_DATE)].pivot(index='date', 
                                                                                         columns='permno', 
                                                                                         values=['ret_combined', 'me', 'dollar_vol'])
# Extract unique PERMNOs from SAS
permnos = data_pivot["ret_combined"].columns.tolist()
permnos = [permno for permno in permnos if permno not in (np.nan, None) and permno != '']

print(f"Working with {len(permnos)} PERMNOs and a total of {data_pivot["ret_combined"].shape[0]} days over the period {params.START_DATE} to {params.END_DATE}.")

df_return = data_pivot["ret_combined"]
df_return.head()

Working with 594 PERMNOs and a total of 1258 days over the period 2020-01-01 to 2024-12-31.


permno,10104,10107,10138,10145,10516,10696,10909,11308,11403,11404,...,92614,92655,92778,93002,93089,93096,93132,93246,93429,93436
date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,0.018309,0.018516,0.027249,0.021412,-0.005178,-0.000951,0.003134,-0.006504,0.029989,-0.017796,...,-0.021571,-0.005034,-0.010003,0.020157,0.011919,0.003590,0.027351,NaN,-0.004000,NaN
2020-01-03,-0.003522,-0.012452,-0.003276,-0.010675,-0.001952,0.007358,0.002852,-0.005456,-0.014278,-0.002588,...,0.006489,-0.010120,-0.005632,-0.025435,0.009727,-0.007346,0.013950,NaN,0.015144,NaN
2020-01-06,0.005208,0.002585,0.006172,-0.007548,-0.007823,0.004812,0.002167,-0.000366,0.006106,-0.003272,...,-0.001901,0.006942,-0.001832,-0.001496,0.002294,-0.003411,0.007104,NaN,-0.009396,NaN
2020-01-07,0.002221,-0.009118,0.003983,0.000563,-0.012046,-0.003848,-0.013110,-0.007682,0.003105,-0.007584,...,-0.006211,-0.006037,0.001001,-0.003443,0.008369,-0.015077,-0.000982,NaN,-0.016391,NaN
2020-01-08,0.003877,0.015928,0.009522,0.000845,-0.011084,0.002747,0.005820,0.001843,0.017026,-0.007984,...,0.004083,0.021084,0.005335,-0.012474,0.009208,-0.004336,0.015909,NaN,-0.010235,NaN
